# FallSense vision/fusion resource gate

Phase 2 是選配。本 notebook 先做資源與 schema gate，不會自動下載 >850GB camera ZIPs，也不會在未對齊 subject/time 前訓練。

In [ ]:
# FIRST EXECUTABLE CELL: 只印資源估算，不 mount Drive、不下載。
RESOURCE_ESTIMATE = {
    "route": "official CameraResizedOF.csv (20x20 optical-flow features)",
    "source_folder_id": "1XDJELfyqXSgjQg-z-s5MHG_YxsSZO5vS",
    "known_download_mb": 120,
    "archive_extract_mb": 0,
    "working_disk_budget_gb": 1.0,
    "ram_budget_gb": 3.0,
    "schema_scan_estimate_min": "2-5 (network excluded)",
    "training_runtime": "NOT ESTIMATED until schema/alignment audit",
    "forbidden": "Subject*Camera*.zip (>850GB total)",
}
print(RESOURCE_ESTIMATE)
USER_CONFIRMED_RESOURCES = False
print("STOPPED AT ESTIMATE" if not USER_CONFIRMED_RESOURCES else "resource gate approved")

In [ ]:
assert USER_CONFIRMED_RESOURCES, (
    "先讓使用者檢查上一格估算；未確認前不可 mount/下載/訓練。"
)
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
SOURCE = Path("/content/drive/MyDrive/FallSense/vision/CameraResizedOF.csv")
assert SOURCE.exists(), (
    "請只將官方 CameraResizedOF.csv 手動放到 Drive；不要放 camera ZIPs。"
)
print({"source": str(SOURCE), "size_mb": SOURCE.stat().st_size / 1_000_000})

In [ ]:
import pandas as pd

preview = pd.read_csv(SOURCE, nrows=20)
print({"columns": list(preview.columns), "preview_shape": preview.shape})
identity_candidates = {"Subject", "Activity", "Trial"}
assert identity_candidates.issubset(preview.columns), (
    "vision CSV 無法直接對應 subject/activity/trial；停止，不得用 row order 猜對齊。"
)
print("schema identity gate PASS; 下一步應先建立與 sensor split manifest 同 checksum 的 mapping")

## Intentional stop

目前沒有經 Colab 實機檢查 `CameraResizedOF.csv` schema，因此不放入可能錯位的訓練碼。若 identity/time mapping 通過，才能以與 sensor run 相同的 LOSO split checksum 新增 vision-only / late-fusion experiments。